# Jaccard Similarity

**Easy** &nbsp;·&nbsp; TensorTonic &nbsp;·&nbsp; `Recommender Systems`

Jaccard similarity measures the overlap between two sets as the ratio of their
intersection over their union. In recommender systems it is used to compare users
by their item interaction histories (purchases, likes, views) **without needing
explicit ratings**. Two users who bought many of the same products will have a
high Jaccard similarity.

Given two lists of items, compute the Jaccard similarity coefficient. Duplicate
items in the input should be treated as a single item (convert to sets first).

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

If both sets are empty, return `0.0`.

---

**Example 1:**

```
Input:  set_a = [1, 2, 3], set_b = [2, 3, 4]
Output: 0.5
intersection = {2, 3} (size 2), union = {1, 2, 3, 4} (size 4). Jaccard = 2/4 = 0.5
```

**Example 2:**

```
Input:  set_a = [1, 2, 3], set_b = [4, 5, 6]
Output: 0.0
no items in common. Intersection is empty, so Jaccard = 0/6 = 0.0
```

---

**Hint 1:** convert both lists to Python sets using `set()`. Then use set
intersection (`&`) and set union (`|`) operators. The Jaccard similarity is
`len(intersection) / len(union)`.

**Hint 2:** handle the edge case where both sets are empty (union has size 0) by
returning `0.0` before dividing.

**Requirements:**

- convert both input lists to sets (remove duplicates)
- compute the intersection and union of the two sets
- return `0.0` if both sets are empty
- return a float between 0.0 and 1.0

**Constraints:**

- input lists may contain duplicates (treat as sets)
- items are integers
- return a float
- time limit 200 ms

### Where the work actually is

No NumPy in this one — it is pure Python sets, and `set()` gives you the
"duplicates count once" requirement for free.

Two things to get right:

1. **The empty case.** `0 / 0` is a `ZeroDivisionError`, not a `nan`. So the
   guard has to come *before* the division. Same lesson as the zero vector in the
   cosine problem — notice it is the same shape of bug twice.
2. **Division in Python 3** already gives you a float, so unlike the NumPy
   problems there is nothing to convert. Check that against the test anyway.

### A second version worth writing

Building the union allocates a whole third set you then only take the `len` of.
There is an identity that gets you `|A ∪ B|` from three numbers you either
already have or can get cheaply:

$$|A \cup B| = |A| + |B| - |A \cap B|$$

Write the version that uses it. Convince yourself the identity is true first —
why the subtraction, and what would go wrong without it?

### And one more, since the names get swapped constantly

Jaccard **distance** is `1 - similarity`. Add it and read the two lines of the
test that check it: similarity `1.0` and distance `0.0` both mean *identical*.
Getting these two backwards is a classic and silent bug.

In [ ]:
class Solution:

    def jaccard_similarity(self, set_a, set_b) -> float:
        """
        Compute the Jaccard similarity between two item sets.
        Returns: float in [0.0, 1.0]   (0.0 if both are empty)
        """
        # TODO
        pass

    # optional: same answer without ever building the union
    def jaccard_similarity_counts(self, set_a, set_b) -> float:
        # TODO
        pass

    # optional: 1 - similarity
    def jaccard_distance(self, set_a, set_b) -> float:
        # TODO
        pass

In [ ]:
def check(got, want, tol=1e-9):
    """Compare one result against its expected value."""
    if got is None:
        return "not implemented"
    try:
        return "OK" if abs(got - want) < tol else f"WRONG got {got!r} want {want!r}"
    except TypeError:
        return f"WRONG got {got!r} (expected a number)"


sol = Solution()

cases = [
    ([1, 2, 3],       [2, 3, 4],     0.5),    # the worked example
    ([1, 2, 3],       [4, 5, 6],     0.0),    # disjoint
    ([1, 2, 3],       [1, 2, 3],     1.0),    # identical
    ([],              [],            0.0),    # both empty -> NOT a crash
    ([1, 2],          [],            0.0),    # one empty
    ([1, 1, 1, 2],    [1, 2, 2],     1.0),    # duplicates collapse -> {1,2} both
    ([1, 2, 3, 4],    [3, 4],        0.5),    # subset
]

print("jaccard_similarity")
for a, b, want in cases:
    print(f"  {str(a):<14} {str(b):<10} -> {check(sol.jaccard_similarity(a, b), want, 1e-12)}")

print("\njaccard_similarity_counts")
for a, b, want in cases:
    print(f"  {str(a):<14} {str(b):<10} -> {check(sol.jaccard_similarity_counts(a, b), want, 1e-12)}")

print("\njaccard_distance   (identical -> 0.0, disjoint -> 1.0)")
print("  identical ->", check(sol.jaccard_distance([1, 2], [1, 2]), 0.0))
print("  disjoint  ->", check(sol.jaccard_distance([1, 2], [3, 4]), 1.0))

### After it passes: what it is for

Four users, four watchlists, **no ratings anywhere** — just "did they watch it".
That is the situation Jaccard exists for.

The cell builds the similarity matrix and then makes an actual recommendation:
find the user most similar to Ari, and suggest what they have seen that Ari has
not. That is collaborative filtering in six lines.

Look at the matrix once it prints, and ask why the diagonal is what it is.

In [ ]:
watchlists = {
    "Ari":  ["Dune", "Her", "Arrival", "Alien"],
    "Bea":  ["Dune", "Her", "Arrival", "Roma"],
    "Cy":   ["Alien", "Dune"],
    "Dev":  ["Amelie", "Roma"],
}

names = list(watchlists)
print(f"{'':<6}" + "".join(f"{n:>8}" for n in names))
for u in names:
    cells = ""
    for v in names:
        s = sol.jaccard_similarity(watchlists[u], watchlists[v])
        cells += f"{s:>8.2f}" if s is not None else f"{'--':>8}"
    print(f"{u:<6}{cells}")

me = "Ari"
if sol.jaccard_similarity(watchlists[me], watchlists["Bea"]) is not None:
    neighbour = max((u for u in names if u != me),
                    key=lambda u: sol.jaccard_similarity(watchlists[me], watchlists[u]))
    unseen = set(watchlists[neighbour]) - set(watchlists[me])
    print(f"\n{me}'s nearest neighbour is {neighbour}")
    print(f"recommend: {sorted(unseen)}")